In [ ]:
import os, glob, json, numpy as np, pandas as pd, pydicom
from PIL import Image
COMP = os.path.dirname(sorted(glob.glob('/kaggle/input/**/train_series.csv', recursive=True), key=len)[0])
PICKS = {'g009': '1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465', 'g018': '1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192', 'g057': '1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312'}
ser = pd.read_csv(os.path.join(COMP, 'train_series.csv'), dtype=str)
os.makedirs('/kaggle/working/png', exist_ok=True)
def norm(a):
    lo, hi = np.percentile(a, [1, 99.5]); return np.clip((a - lo) / (hi - lo + 1e-6) * 255, 0, 255).astype(np.uint8)
for tag, uid in PICKS.items():
    for _, r in ser[ser.StudyInstanceUID == uid].iterrows():
        d = os.path.join(COMP, 'train_series', uid, r['SeriesInstanceUID'])
        fs = glob.glob(d + '/*.dcm') or glob.glob(d + '/**/*', recursive=True)
        sl = []
        for f in fs:
            try:
                ds = pydicom.dcmread(f); z = float(getattr(ds, 'InstanceNumber', 0)); sl.append((z, ds.pixel_array.astype(np.float32)))
            except Exception as e:
                pass
        if not sl: print('no pixels', tag, r['SeriesInstanceUID'][-8:]); continue
        sl.sort(key=lambda x: x[0]); n = len(sl); idx = np.linspace(int(n * .2), int(n * .8) - 1, 9).astype(int)
        tiles = [Image.fromarray(norm(sl[i][1])).resize((256, 256)) for i in idx]
        m = Image.new('L', (768, 768))
        for k, im in enumerate(tiles): m.paste(im, ((k % 3) * 256, (k // 3) * 256))
        name = '%s_%s_%s_%s.png' % (tag, str(r.get('Anatomical_Plane', 'x')), str(r.get('Fat_Suppression', 'x')), r['SeriesInstanceUID'][-6:])
        m.save('/kaggle/working/png/' + name.replace(' ', ''))
        print('saved', name, 'slices', n)
